In [5]:
import pickle
import pandas as pd

raw_train = pickle.load(open("../../data_processing/splits/train.pkl", "rb"))

In [48]:
# Compile into a single dataframe
train_df = pd.concat(
    [df.assign(home_id=home_id, has_ev=has_ev, city=city)
     for home_id, (has_ev, city, df) in raw_train.items()]
).reset_index()

# Rename columns
train_df.rename(columns={"car1": "ev_load", "load": "total_load"}, inplace=True)

# Split day-time into two columns
train_df["day"] = train_df["localminute"].dt.normalize()
train_df["time"] = train_df["localminute"].dt.strftime("%H:%M")
train_df = train_df.drop(columns="localminute")

train_df['time'] = pd.to_timedelta(train_df['time'] + ':00') # convert to time delta
train_df['time_index'] = (
    train_df['time'].dt.total_seconds() // (15 * 60)
).astype(int) # 0-95 time index

# Change types
train_df["charge_state"] = train_df["charge_state"].astype("int")

print(f"Rows before filtering out incomplete days: {len(train_df)}")

# Keep only (home_id, day) groups that have all 96 time indices (0–95)
full_day_mask = (
    train_df.groupby(["home_id", "day"])["time_index"]
    .transform(lambda x: x.nunique() == 96)
)
train_df = train_df[full_day_mask]

# Count number of complete days tracked per home
days_per_home = (
    train_df.groupby("home_id")["day"]
    .nunique()
    .rename("num_days")
    .reset_index()
)
print(f"Rows after filtering out incomplete days: {len(train_df)}")
print(f"Complete (home_id, day) pairs remaining: {len(train_df) // 96}")
days_per_home

Rows before filtering out incomplete days: 1559407
Rows after filtering out incomplete days: 1543872
Complete (home_id, day) pairs remaining: 16082


,home_id,num_days
0,27,183
1,203,360
2,387,183
3,558,183
4,661,350
5,914,183
6,950,183
7,1240,183
8,1450,360
9,1524,360
